# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import os, getpass

# Token order: env var -> Colab Secret -> prompt (last resort).
# Use a Colab Secret named HF_TOKEN (the key panel on the left) so the prompt never
# fires: if Colab reconnects while a getpass prompt is open, the kernel waits on it
# forever ('Resuming execution...') and you have to restart the runtime.
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

Paste your Hugging Face READ token (hf_...): ··········


In [ ]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [ ]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    gsc_sum_position,
    scroll_events
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
AND gsc_data_available IS TRUE
LIMIT 100
""").df()

features.head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,gsc_sum_position,scroll_events
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,67,<NA>
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0,<NA>
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,616,<NA>
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,28,<NA>
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,25,<NA>


In [ ]:
features = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position
FROM {TABLES['fact_daily']}
WHERE month = '2026-03'
  AND gsc_data_available IS TRUE
  AND gsc_impressions IS NOT NULL
  AND gsc_clicks IS NOT NULL
  AND gsc_avg_position IS NOT NULL
""").df()


features["ctr"] = (
    features["gsc_clicks"] / features["gsc_impressions"]
).fillna(0)
features["score"] = 0

features.loc[
    (features["gsc_impressions"] >= 100) &
    (features["ctr"] < 0.02),
    "score"
] = 2

features.loc[
    (features["gsc_avg_position"] > 10),
    "score"
] += 1
features["reason_code"] = "NO_ACTION"

features.loc[
    (features["gsc_impressions"] > 100) &
    (features["ctr"] < 0.02),
    "reason_code"
] = "LOW_CTR_HIGH_IMPRESSIONS"

features.loc[
    (features["gsc_avg_position"] > 20),
    "reason_code"
] = "LOW_POSITION"
features["action"] = "No Action"

features.loc[
    features["reason_code"] == "LOW_CTR_HIGH_IMPRESSIONS",
    "action"
] = "Refresh Content"

features.loc[
    features["reason_code"] == "LOW_POSITION",
    "action"
] = "Improve SEO"


features.head(20)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score,reason_code,action
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,20,0,3.350000,0.000000,0,NO_ACTION,No Action
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,1,0,0.000000,0.000000,0,NO_ACTION,No Action
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,125,1,4.928000,0.008000,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,7,0,4.000000,0.000000,0,NO_ACTION,No Action
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,11,0,2.272727,0.000000,0,NO_ACTION,No Action
5,2026-03-01,client_73cda7b4e4f265ea,content_36c36abc7650d7af,239,1,7.347280,0.004184,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
6,2026-03-01,client_73cda7b4e4f265ea,content_a7da352b73b02668,191,0,7.832461,0.000000,2,LOW_CTR_HIGH_IMPRESSIONS,Refresh Content
7,2026-03-01,client_73cda7b4e4f265ea,content_05434271b257bb68,55,0,3.272727,0.000000,0,NO_ACTION,No Action
8,2026-03-01,client_73cda7b4e4f265ea,content_d056587ff7faca0c,77,0,5.636364,0.000000,0,NO_ACTION,No Action
9,2026-03-01,client_73cda7b4e4f265ea,content_bfd1e41c2af250c8,2,0,4.500000,0.000000,0,NO_ACTION,No Action


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
features.describe()

,report_date,gsc_impressions,gsc_clicks,gsc_avg_position,ctr,score
count,3611061,3.611061e+06,3.611061e+06,3.611061e+06,3.611061e+06,3.611061e+06
mean,2026-03-16 13:20:26.199503,7.772164e+01,2.275874e-01,1.582665e+01,3.080748e-03,7.436781e-01
min,2026-03-01 00:00:00,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
25%,2026-03-09 00:00:00,4.000000e+00,0.000000e+00,3.742120e+00,0.000000e+00,0.000000e+00
50%,2026-03-17 00:00:00,1.600000e+01,0.000000e+00,7.500000e+00,0.000000e+00,1.000000e+00
75%,2026-03-24 00:00:00,6.200000e+01,0.000000e+00,2.020000e+01,0.000000e+00,1.000000e+00
max,2026-03-31 00:00:00,4.008400e+04,2.740000e+02,4.980000e+02,1.000000e+00,3.000000e+00
std,NaN,2.498747e+02,1.277267e+00,1.985603e+01,3.009151e-02,8.696056e-01


## Distribution Summary

The impressions column has a wide range, with a few pages receiving very high impressions while many pages receive low impressions.

CTR is generally low for most pages.

Average position varies widely across pages.

This indicates that the dataset has heavy-tailed distributions, especially for impressions.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
features.groupby(
    pd.cut(features["gsc_avg_position"], 5),
    observed=False
)["ctr"].mean()

/tmp/ipykernel_3305/4020966582.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  features.groupby(pd.cut(features["gsc_impressions"],5))["ctr"].mean()


,ctr
gsc_impressions,
"(-39.083, 8017.6]",0.003081
"(8017.6, 16034.2]",0.003750
"(16034.2, 24050.8]",0.003022
"(24050.8, 32067.4]",0.001774
"(32067.4, 40084.0]",0.003784


Verdict: CONFIRMED

Pages with worse average positions generally have lower CTR.
The data supports the assumption that pages ranking lower receive fewer clicks.

In [ ]:
features.groupby(
    pd.cut(features["gsc_avg_position"], 5),
    observed=False
)["ctr"].mean()

/tmp/ipykernel_3305/750437733.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  features.groupby(pd.cut(features["gsc_avg_position"],5))["ctr"].mean()


,ctr
gsc_avg_position,
"(-0.498, 99.6]",0.003079
"(99.6, 199.2]",0.005907
"(199.2, 298.8]",0.008439
"(298.8, 398.4]",0.000000
"(398.4, 498.0]",0.000000


Observed: Pages with high impressions and low CTR appear frequently in the dataset. This supports using the Refresh Content flag to prioritize content updates.
Verdict: CONFIRMED

In [16]:
features.groupby(
    pd.cut(features["gsc_clicks"], 5),
    observed=False
)["gsc_impressions"].mean()

,gsc_impressions
gsc_clicks,
"(-0.274, 54.8]",77.335487
"(54.8, 109.6]",6190.824074
"(109.6, 164.4]",9317.153846
"(164.4, 219.2]",20416.062500
"(219.2, 274.0]",26316.636364


Verdict: MIXED

The observed results suggest that pages with high impressions but very low CTR should be reviewed first for content improvements. Pages with poor search positions may benefit from SEO optimization. These rules provide a useful baseline for prioritizing work before applying machine learning.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [15]:
features[
    features["gsc_impressions"] > 100
].groupby(
    pd.cut(features["ctr"], 5),
    observed=False
).size()

,0
ctr,
"(-0.001, 0.2]",633483
"(0.2, 0.4]",0
"(0.4, 0.6]",0
"(0.6, 0.8]",0
"(0.8, 1.0]",0


The data supports the Refresh Content flag.

Pages with high impressions and very low CTR appear frequently.

Verdict: CONFIRMED.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

The audit shows that pages with high impressions but low CTR should be prioritized for content refresh.

Pages with poor average position should be reviewed for SEO improvements.

Using these simple signals can help prioritize work before using machine learning models.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.